# Introduction générale 

Les inégalités salariales entre les femmes et les hommes constituent un enjeu important du marché du travail. Malgré les évolutions observées au cours des dernières décennies, des différences de rémunération persistent entre les deux sexes. Toutefois, ces écarts ne sont pas nécessairement homogènes : ils peuvent varier selon la catégorie socioprofessionnelle, le niveau de rémunération ou encore la période considérée.

L'objectif de cette étude est d'analyser l'évolution et la structure des écarts salariaux entre les femmes et les hommes dans le secteur privé français, sur la période 1996-2024. Pour cela, nous mobilisons plusieurs spécifications économétriques complémentaires. La première vise à étudier l'évolution de l'écart salarial moyen dans le temps. La deuxième cherche à identifier les différences observées selon les catégories socioprofessionnelles (CSP), tandis que la troisième examine la répartition de ces écarts selon différents niveaux de la distribution des salaires.

Cette approche permet ainsi de dépasser une simple comparaison des salaires moyens et de mettre en évidence l'hétérogénéité des inégalités salariales. Il convient néanmoins de préciser que les données utilisées sont agrégées et portent sur les salaires mensuels, tous temps de travail confondus. Les résultats doivent donc être interprétés comme des écarts salariaux observés et non comme une mesure directe de l'effet causal du sexe sur la rémunération.

## 1. Chargement et préparation des données


In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

prive = pd.read_csv(
    "Data nettoyees/DS_DERA_PRIVE_SERIES_LONGUES_data_nettoye.csv",
    sep=";",
    encoding="utf-8-sig"
)

SALAIRE = "Salaire net moyen mensuel (Euros constants)"

## 2. Construction de l’échantillon

On conserve le salaire net moyen mensuel, pour la France et l’ensemble des secteurs, âges, tailles d’entreprise et temps de travail. On écarte les modalités « Ensemble » du sexe et de la CSP afin de comparer les femmes et les hommes dans chacune des CSP.

In [18]:
# Conditions communes
base = (
    prive["indicateur"].eq(SALAIRE)
    & prive["centile_salaire"].eq("Ensemble")
    & prive["geo"].eq("F")
    & prive["secteur_activite"].eq("Ensemble des secteurs")
    & prive["tranche_age"].eq("Ensemble")
    & prive["taille_entreprise"].eq("Ensemble")
    & prive["temps_travail"].eq("Ensemble")
    & prive["sexe"].isin(["Femme", "Homme"])
)

# Échantillon CSP
echantillon_csp = prive.loc[
    base & prive["categorie_socio_prof"].ne("Ensemble"),
    ["annee", "sexe", "categorie_socio_prof", "valeur"]
].copy()

# Échantillon global : évolution temporelle
echantillon_tendance = prive.loc[
    base & prive["categorie_socio_prof"].eq("Ensemble"),
    ["annee", "sexe", "valeur"]
].copy()

# Échantillon par quantile
quantiles = [
    "10e centile (D1 - 10% gagnent moins)",
    "1er quartile (Q1 - 25% gagnent moins)",
    "Médiane (50% gagnent moins/plus)",
    "3e quartile (Q3 - 75% gagnent moins)",
    "90e centile (D9 - 10% gagnent plus)",
    "95e centile (Top 5%)",
    "99e centile (Top 1%)"
]

echantillon_quantiles = prive.loc[
    prive["indicateur"].eq(SALAIRE)
    & prive["centile_salaire"].isin(quantiles)
    & prive["geo"].eq("F")
    & prive["secteur_activite"].eq("Ensemble des secteurs")
    & prive["tranche_age"].eq("Ensemble")
    & prive["taille_entreprise"].eq("Ensemble")
    & prive["temps_travail"].eq("Ensemble")
    & prive["categorie_socio_prof"].eq("Ensemble")
    & prive["sexe"].isin(["Femme", "Homme"]),
    ["annee", "sexe", "centile_salaire", "valeur"]
].copy()

# Préparation commune des variables
for df in [echantillon_csp, echantillon_tendance, echantillon_quantiles]:
    df["annee"] = pd.to_numeric(df["annee"])
    df["valeur"] = pd.to_numeric(df["valeur"])

    df["femme"] = (df["sexe"] == "Femme").astype(int)

import numpy as np
import pandas as pd

def calculer_ecarts(modele, variable, categories, reference):
    resultats = []

    for categorie in categories:
        if categorie == reference:
            continue

        # Coefficient associé à la catégorie
        coef_csp = modele.params.get(
            f"C({variable}, Treatment(reference='{reference}'))[T.{categorie}]",
            0
        )

        # Coefficient d'interaction Femme × catégorie
        coef_interaction = modele.params.get(
            f"femme:C({variable}, Treatment(reference='{reference}'))[T.{categorie}]",
            0
        )

        # Écart total femmes-hommes pour cette catégorie
        ecart_log = modele.params["femme"] + coef_interaction

        # Conversion en pourcentage
        ecart_pct = 100 * (np.exp(ecart_log) - 1)

        resultats.append({
            "CSP": categorie,
            "Écart salarial (%)": ecart_pct
        })

    # Ajouter la catégorie de référence
    ecart_reference = 100 * (np.exp(modele.params["femme"]) - 1)

    resultats.append({
        "CSP": reference,
        "Écart salarial (%)": ecart_reference
    })

    return pd.DataFrame(resultats)

## 3. Régression n° 1 – L’évolution de l’écart salarial dans le temps

Dans un premier temps, nous cherchons à déterminer si les écarts de rémunération entre les femmes et les hommes se sont réduits au cours des dernières décennies. Pour cela, nous construisons un indicateur correspondant au logarithme du rapport entre le salaire net moyen mensuel des femmes et celui des hommes. Cet indicateur est ensuite régressé sur une tendance temporelle linéaire, afin d'identifier l'évolution moyenne de cet écart sur la période étudiée.


Soit, sous forme mathématique :

$$

\boxed{
\ln\left(\frac{SalaireFemmes_t}{SalaireHommes_t}\right)
=\alpha+\beta \times trend_t+\varepsilon_t
}

$$
Avec :



`trend_t` : tendance temporelle, égale à 0 en 1996 et à 28 en 2024 ;

`alpha` : constante, correspondant à l'écart estimé en 1996 ;

`beta` : évolution annuelle moyenne de l'écart salarial ;

`epsilon_t` : terme d'erreur.












In [19]:
salaire = echantillon_tendance.pivot(
    index="annee", columns="sexe", values="valeur"
)

ecarts = pd.DataFrame({
    "ecart": np.log(salaire["Femme"] / salaire["Homme"])
})

ecarts["trend"] = ecarts.index - 1996

modele_tendance = smf.ols(
    "ecart ~ trend", data=ecarts
).fit(cov_type="HAC", cov_kwds={"maxlags": 1})

print(modele_tendance.summary())

# Interprétation en pourcentage
for terme in ["Intercept", "trend"]:
    beta = modele_tendance.params[terme]
    ic = modele_tendance.conf_int().loc[terme]

    print(
        terme,
        f": {100*np.expm1(beta):.2f} %",
        f"(IC 95 % : {100*np.expm1(ic.iloc[0]):.2f} % ; "
        f"{100*np.expm1(ic.iloc[1]):.2f} %)"
    )

                            OLS Regression Results                            
Dep. Variable:                  ecart   R-squared:                       0.932
Model:                            OLS   Adj. R-squared:                  0.929
Method:                 Least Squares   F-statistic:                     170.8
Date:                Sun, 22 Nov 2026   Prob (F-statistic):           3.44e-13
Time:                        22:02:52   Log-Likelihood:                 96.735
No. Observations:                  29   AIC:                            -189.5
Df Residuals:                      27   BIC:                            -186.7
Df Model:                           1                                         
Covariance Type:                  HAC                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -0.2590      0.004    -60.070      0.0

Les résultats mettent en évidence un écart salarial initial important. En 1996, le salaire net moyen mensuel des femmes était inférieur de 22,82 % à celui des hommes. Ce résultat est statistiquement significatif et témoigne de différences de rémunération déjà marquées au début de la période.

Toutefois, le coefficient associé à la tendance temporelle est positif et significatif. Il indique que le rapport entre les salaires moyens des femmes et des hommes augmente d'environ 0,38 % par an. Autrement dit, les rémunérations féminines se rapprochent progressivement de celles des hommes. Selon les prédictions du modèle, l'écart salarial passerait ainsi de 22,82 % en 1996 à environ 14,1 % en 2024. Cette évolution traduit une amélioration notable sur le long terme, même si les différences de rémunération restent importantes.

Enfin, le pouvoir explicatif du modèle est relativement élevé, avec un R² de 0,932. La tendance temporelle permet donc de rendre compte d'une grande partie de la variation observée de l'indicateur salarial. Il convient néanmoins de nuancer ce résultat au regard de la forte autocorrélation positive des résidus, suggérée par une statistique de Durbin-Watson de 0,171. Malgré l'utilisation d'erreurs-types robustes de Newey-West, des tests de sensibilité au choix du nombre de retards seraient utiles pour vérifier la robustesse de l'inférence statistique.



## 2. Hétérogénéité des écarts salariaux selon la catégorie socioprofessionnelle

Le modèle utilise les salaires nets moyens mensuels par sexe et CSP, en logarithme, avec effets fixes d’année. Les ouvriers et les hommes sont les catégories de référence :

Dans un second temps, nous cherchons à déterminer si les écarts de rémunération varient selon la catégorie socioprofessionnelle. Pour cela, nous estimons une régression linéaire intégrant des interactions entre le sexe et la CSP, ainsi que des effets fixes annuels.



$$\boxed{
\ln(Salaire_{it}) =
\alpha+\beta_1 Femme_i
+\sum_j\gamma_j CSP_{ij}
+\sum_j\delta_j(Femme_i\times CSP_{ij})
+\lambda_t+\varepsilon_{it}
}$$

 Avec :

`Femme_i` : variable indicatrice égale à 1 pour les femmes et 0 pour les hommes ;

`CSP_ij` : variables indicatrices des catégories socioprofessionnelles ;

`gamma_j` : différences salariales entre les CSP chez les hommes ;

`delta_j` : différences d'écarts salariaux femmes-hommes selon la CSP ;

`lambda_t` : effets fixes annuels ;

`epsilon_it` : terme d'erreur.



In [20]:
modele_csp = smf.ols(
    "np.log(valeur) ~ femme * C(categorie_socio_prof, Treatment(reference='Ouvriers')) + C(annee)",
    data=echantillon_csp
).fit(
    cov_type="cluster",
    cov_kwds={"groups": echantillon_csp["annee"], "use_correction": True}
)

print(modele_csp.summary())

csp = sorted(echantillon_csp["categorie_socio_prof"].unique())

table_csp = calculer_ecarts(
    modele_csp, "categorie_socio_prof", csp, "Ouvriers"
)

display(table_csp.round(3))

                            OLS Regression Results                            
Dep. Variable:         np.log(valeur)   R-squared:                       0.995
Model:                            OLS   Adj. R-squared:                  0.994
Method:                 Least Squares   F-statistic:                 1.793e+04
Date:                Sun, 22 Nov 2026   Prob (F-statistic):           2.22e-49
Time:                        22:03:18   Log-Likelihood:                 518.51
No. Observations:                 232   AIC:                            -965.0
Df Residuals:                     196   BIC:                            -840.9
Df Model:                          35                                         
Covariance Type:              cluster                                         
                                                                                                             coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------

/usr/local/python/3.14.2/lib/python3.14/site-packages/statsmodels/base/model.py:1988: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 35, but rank is 7
  res = self.wald_test(


,CSP,Écart salarial (%)
0,Cadres et prof. intellectuelles sup.,-22.111
1,Employés,-7.365
2,Professions intermédiaires,-13.756
3,Ouvriers,-16.694


Les résultats font d'abord apparaître des différences importantes de rémunération entre les catégories socioprofessionnelles. Chez les hommes, les cadres perçoivent un salaire mensuel moyen environ 158,5 % supérieur à celui des ouvriers, tandis que l'écart atteint environ 42,7 % pour les professions intermédiaires. En revanche, aucune différence statistiquement significative n'est mise en évidence entre les employés et les ouvriers dans cette spécification.

Au-delà de ces différences entre catégories, les coefficients d'interaction permettent de mettre en évidence une forte hétérogénéité des écarts femmes-hommes. Dans la catégorie de référence, les ouvriers, les femmes perçoivent un salaire mensuel moyen inférieur de 16,69 % à celui des hommes. Cet écart est encore plus important chez les cadres, où il atteint 22,11 %. À l'inverse, il est plus limité chez les professions intermédiaires (-13,76 %) et surtout chez les employés (-7,37 %).

Ces résultats montrent que les inégalités salariales ne sont pas uniformément réparties sur le marché du travail. L'écart observé est particulièrement marqué chez les cadres, alors qu'il est relativement plus faible chez les employés. Les interactions étant statistiquement significatives, les différences d'écarts entre les catégories et la catégorie de référence sont statistiquement marquées dans le modèle.

Il faut toutefois rester prudent dans l'interprétation de ces résultats. La catégorie socioprofessionnelle ne permet pas, à elle seule, de contrôler l'ensemble des caractéristiques des emplois occupés. Les différences observées peuvent également refléter des écarts d'ancienneté, de responsabilités, de métiers exercés ou de durée du travail. Il ne s'agit donc pas d'un effet causal de la CSP ou du sexe sur les salaires, mais de différences conditionnelles observées entre groupes.

## 5.  Écarts salariaux selon la position dans la distribution des rémunérations

Enfin, la troisième régression vise à étudier la manière dont les écarts salariaux évoluent selon la position des individus dans la distribution des rémunérations. En effet, une comparaison des seuls salaires moyens peut masquer des disparités importantes entre les salariés les moins rémunérés et ceux qui se situent au sommet de la distribution. Nous mobilisons donc différents quantiles, du premier décile (D1) au 99e centile (P99).

Le modèle est :

$$\boxed{
\ln(Salaire_{itq}) =
\alpha+\beta_1 Femme_i
+\sum_q\gamma_q Quantile_q
+\sum_q\delta_q(Femme_i\times Quantile_q)
+\lambda_t+\varepsilon_{itq}
}$$

Avec :

`Femme_i` : variable indicatrice du sexe ;

`Quantile_q` : variables indicatrices des différents quantiles de salaire ;

`gamma_q` : différences salariales entre les quantiles chez les hommes ;

`delta_q` : différences d'écarts femmes-hommes selon le quantile ;

`lambda_t` : effets fixes annuels ;

`epsilon_itq` : terme d'erreur.

La médiane et les hommes constituent respectivement les catégories de référence pour le quantile et le sexe.

In [21]:
reference = "Médiane (50% gagnent moins/plus)"

modele_quantiles = smf.ols(
    f"np.log(valeur) ~ femme * C(centile_salaire, Treatment(reference='{reference}')) + C(annee)",
    data=echantillon_quantiles
).fit(
    cov_type="cluster",
    cov_kwds={"groups": echantillon_quantiles["annee"], "use_correction": True}
)

print(modele_quantiles.summary())

table_quantiles = calculer_ecarts(
    modele_quantiles, "centile_salaire", quantiles, reference
)

display(table_quantiles.round(3))

                            OLS Regression Results                            
Dep. Variable:         np.log(valeur)   R-squared:                       0.999
Model:                            OLS   Adj. R-squared:                  0.998
Method:                 Least Squares   F-statistic:                 4.921e+04
Date:                Sun, 22 Nov 2026   Prob (F-statistic):           2.04e-57
Time:                        22:03:31   Log-Likelihood:                 940.31
No. Observations:                 406   AIC:                            -1797.
Df Residuals:                     364   BIC:                            -1628.
Df Model:                          41                                         
Covariance Type:              cluster                                         
                                                                                                                                 coef    std err          z      P>|z|      [0.025      0.975]
-------------------

/usr/local/python/3.14.2/lib/python3.14/site-packages/statsmodels/base/model.py:1988: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 41, but rank is 13
  res = self.wald_test(


,CSP,Écart salarial (%)
0,10e centile (D1 - 10% gagnent moins),-6.282
1,1er quartile (Q1 - 25% gagnent moins),-9.610
2,3e quartile (Q3 - 75% gagnent moins),-15.986
3,90e centile (D9 - 10% gagnent plus),-22.786
4,95e centile (Top 5%),-27.157
5,99e centile (Top 1%),-34.981
6,Médiane (50% gagnent moins/plus),-12.972


Les résultats révèlent une progression particulièrement nette des écarts salariaux à mesure que l'on s'élève dans la distribution des rémunérations. Au premier décile, le salaire des femmes est inférieur de 6,28 % à celui des hommes. Cet écart atteint 12,97 % à la médiane, puis s'accentue progressivement pour atteindre 22,79 % au 90e centile et près de 35 % au 99e centile. Les différences estimées sont statistiquement significatives pour l'ensemble des quantiles étudiés.

Cette évolution met en évidence une concentration des écarts salariaux les plus importants dans le haut de la distribution. Alors que les différences de rémunération restent relativement limitées parmi les salariés les moins rémunérés, elles deviennent nettement plus marquées parmi les salariés les mieux rémunérés. Le résultat obtenu au 99e centile est particulièrement significatif d'un point de vue économique : les femmes situées à ce niveau de leur distribution salariale perçoivent un salaire inférieur de près de 35 % à celui des hommes situés au même quantile.

Ces résultats peuvent notamment conduire à s'interroger sur les différences d'accès aux postes de direction, aux fonctions à forte responsabilité ou aux emplois offrant les rémunérations les plus élevées. Ils suggèrent que les inégalités salariales ne se limitent pas au bas de l'échelle des rémunérations et sont particulièrement prononcées dans sa partie supérieure.

Il convient cependant de souligner que notre modèle ne permet pas d'identifier directement les mécanismes à l'origine de ces différences. Par ailleurs, cette estimation constitue une régression linéaire sur des valeurs agrégées de quantiles et non une régression quantile au sens économétrique strict.

## 6. Conclusion 

Les trois régressions permettent de dégager plusieurs enseignements complémentaires sur les inégalités salariales entre les femmes et les hommes dans le secteur privé français.

Tout d'abord, l'analyse temporelle met en évidence une réduction significative de l'écart salarial moyen depuis 1996, même si celui-ci demeure important en 2024. Ensuite, les résultats montrent que ces écarts varient sensiblement selon la catégorie socioprofessionnelle, avec une différence particulièrement marquée chez les cadres. Enfin, l'analyse par quantiles révèle une accentuation progressive des écarts à mesure que l'on se rapproche du sommet de la distribution des salaires.

Ces résultats soulignent ainsi que les inégalités salariales ne peuvent être résumées par un indicateur moyen unique. Leur ampleur dépend à la fois de la position professionnelle et du niveau de rémunération considéré. Les deux dernières régressions font notamment apparaître des écarts plus importants dans certaines catégories et dans le haut de la distribution. Il faut toutefois éviter de conclure que ces deux phénomènes concernent nécessairement les mêmes individus, puisqu'aucune spécification ne croise simultanément la CSP et les quantiles.

Plusieurs limites doivent enfin être prises en compte. Premièrement, les données utilisées sont agrégées et ne permettent pas de contrôler les caractéristiques individuelles des salariés, telles que le diplôme, l'expérience professionnelle, l'ancienneté ou le métier exercé. Deuxièmement, les salaires considérés sont des salaires mensuels pour l'ensemble des temps de travail, et non des salaires en équivalent temps plein. Une partie des écarts observés peut donc provenir de différences de durée du travail, notamment du recours au temps partiel. Enfin, les résultats reposent sur des associations statistiques et ne permettent pas d'établir un lien de causalité entre le sexe et le niveau de rémunération.

En définitive, cette étude met en évidence une réduction des écarts salariaux entre les femmes et les hommes sur longue période, mais également leur persistance et leur forte hétérogénéité selon la position professionnelle et salariale.


## Remarques méthodologiques complémentaires

Pour renforcer la qualité économétrique du travail, plusieurs précautions doivent être conservées :

Absence d'interprétation causale : les données sont agrégées et ne permettent pas de contrôler les caractéristiques individuelles des salariés. Les résultats doivent donc être présentés comme des associations statistiques et des écarts observés.

Nature de la troisième régression : il ne s'agit pas d'une régression quantile au sens de Koenker et Bassett, mais d'une régression linéaire appliquée à des valeurs agrégées de quantiles.

Interprétation des R² : les valeurs élevées obtenues dans les deuxième et troisième régressions s'expliquent en partie par les effets fixes annuels et les différences de niveaux de salaire entre catégories. Elles ne démontrent pas que les modèles identifient les causes des inégalités.

Autocorrélation temporelle : le Durbin-Watson de 0,171 dans la première régression suggère une forte autocorrélation positive des résidus. Il est donc recommandé de vérifier la sensibilité des erreurs-types Newey-West à différents nombres de retards (maxlags = 1, 2, 3 ou 4).

Erreurs-types clusterisées : les deuxième et troisième régressions reposent sur 29 groupes temporels. Ce nombre relativement limité invite à interpréter avec prudence les p-values et les intervalles de confiance.